In [ ]:
import pandas as pd
import numpy as np
import math
import re
from collections import Counter


df_raw = pd.read_csv('data.csv', on_bad_lines='skip')


df_200k = df_raw.dropna().head(200000).copy()

def calculate_entropy(password):
    if not password: return 0
    p, lns = Counter(str(password)), float(len(str(password)))
    return -sum(count/lns * math.log2(count/lns) for count in p.values())

def extract_features(pwd):
    pwd = str(pwd)
    ln = len(pwd)
    # Basic Counts
    u, l, d = sum(1 for c in pwd if c.isupper()), sum(1 for c in pwd if c.islower()), sum(1 for c in pwd if c.isdigit())
    s = ln - (u + l + d)
    unq = len(set(pwd))
    
    
    return pd.Series({
        'length': ln, 'digits': d, 'upper': u, 'lower': l, 'special': s, 'unique_chars': unq,
        'digit_ratio': d/ln if ln > 0 else 0, 'upper_ratio': u/ln if ln > 0 else 0,
        'lower_ratio': l/ln if ln > 0 else 0, 'special_ratio': s/ln if ln > 0 else 0,
        'has_digit': 1 if d > 0 else 0, 'has_upper': 1 if u > 0 else 0,
        'has_lower': 1 if l > 0 else 0, 'has_special': 1 if s > 0 else 0,
        'starts_with_letter': 1 if pwd[0:1].isalpha() else 0, 'ends_with_digit': 1 if pwd[-1:].isdigit() else 0,
        'repeated_chars': ln - unq, 'consecutive_digits': len(re.findall(r'\d{2,}', pwd)),
        'entropy': calculate_entropy(pwd), 'char_type_count': sum([1 for x in [u, l, d, s] if x > 0]),
        'complexity_score': (ln * 0.1) + (unq * 0.2) + (s * 0.5),
        'length_category': 0 if ln < 8 else (1 if ln < 12 else 2),
        'char_diversity': unq / ln if ln > 0 else 0, 'contains_pattern': 1 if re.search(r'(123|abc|qwerty)', pwd.lower()) else 0
    })

print(f"Original rows: {len(df_raw)} | Slicing to: {len(df_200k)}")


feature_df = df_200k['password'].apply(extract_features)
final_dataset = pd.concat([df_200k, feature_df], axis=1)


final_dataset.to_csv('password_features_dataset.csv', index=False)


Original rows: 669640 | Slicing to: 200000


In [ ]:

df = final_dataset


pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)


df.head()

,password,strength,length,digits,upper,lower,special,unique_chars,digit_ratio,upper_ratio,lower_ratio,special_ratio,has_digit,has_upper,has_lower,has_special,starts_with_letter,ends_with_digit,repeated_chars,consecutive_digits,entropy,char_type_count,complexity_score,length_category,char_diversity,contains_pattern
0,kzde5577,1,8.0,4.0,0.0,4.0,0.0,6.0,0.500000,0.0,0.500000,0.0,1.0,0.0,1.0,0.0,1.0,1.0,2.0,1.0,2.500000,2.0,2.0,1.0,0.750000,0.0
1,kino3434,1,8.0,4.0,0.0,4.0,0.0,6.0,0.500000,0.0,0.500000,0.0,1.0,0.0,1.0,0.0,1.0,1.0,2.0,1.0,2.500000,2.0,2.0,1.0,0.750000,0.0
2,visi7k1yr,1,9.0,2.0,0.0,7.0,0.0,8.0,0.222222,0.0,0.777778,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,2.947703,2.0,2.5,1.0,0.888889,0.0
3,megzy123,1,8.0,3.0,0.0,5.0,0.0,8.0,0.375000,0.0,0.625000,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,3.000000,2.0,2.4,1.0,1.000000,1.0
4,lamborghin1,1,11.0,1.0,0.0,10.0,0.0,11.0,0.090909,0.0,0.909091,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,3.459432,2.0,3.3,1.0,1.000000,0.0


In [ ]:
df.columns

Index(['password', 'strength', 'length', 'digits', 'upper', 'lower', 'special',
       'unique_chars', 'digit_ratio', 'upper_ratio', 'lower_ratio',
       'special_ratio', 'has_digit', 'has_upper', 'has_lower', 'has_special',
       'starts_with_letter', 'ends_with_digit', 'repeated_chars',
       'consecutive_digits', 'entropy', 'char_type_count', 'complexity_score',
       'length_category', 'char_diversity', 'contains_pattern'],
      dtype='object')

In [ ]:
df.shape

(200000, 26)